# 68206 Validation Notebook

This notebook reloads `68206_Pipeline.pkl` and validates it on `complaints_modeltesting100.csv` using the same `feature_engineering.py` logic used in training.


In [1]:
import pickle
import pandas as pd
from feature_engineering import feature_engineering

MODEL_PATH = "68206_Pipeline.pkl"
VALIDATION_DATA_PATH = "complaints_modeltesting100.csv"
OUTPUT_PATH = "68206_validation_predictions.csv"
SAVE_PREDICTIONS = False

# Load saved model
with open(MODEL_PATH, "rb") as f:
    loaded_model = pickle.load(f)

# Load external validation dataset
validation_df = pd.read_csv(VALIDATION_DATA_PATH)
print(f"Validation dataset shape: {validation_df.shape}")

# Apply the same feature engineering used in training
X_validation = feature_engineering(validation_df)
print(f"Engineered feature shape: {X_validation.shape}")

# Generate predictions
predictions = loaded_model.predict(X_validation)
predictions_series = pd.Series(predictions, name="prediction")

# Verify predictions are strictly binary (0 and 1)
unique_values = set(predictions_series.dropna().astype(int).unique())
invalid_values = unique_values - {0, 1}
if invalid_values:
    raise ValueError(f"Non-binary predictions found: {invalid_values}")

print("Prediction value counts:")
print(predictions_series.value_counts().sort_index())
print("First 10 predictions:")
print(predictions_series.head(10).to_list())
print("Binary check passed: predictions contain only 0 and 1.")

# Optionally save predictions
if SAVE_PREDICTIONS:
    output_df = validation_df.copy()
    output_df["prediction"] = predictions_series
    output_df.to_csv(OUTPUT_PATH, index=False)
    print(f"Saved predictions to {OUTPUT_PATH}")


Validation dataset shape: (100, 18)
Engineered feature shape: (100, 16)
Prediction value counts:
prediction
0    31
1    69
Name: count, dtype: int64
First 10 predictions:
[0, 1, 0, 1, 1, 1, 1, 0, 0, 1]
Binary check passed: predictions contain only 0 and 1.
